# 🗣️ Translate with attention: English ↔ Hinglish

Welcome to the **attention** challenge! Today you build the model that kicked off the attention era:
an **encoder–decoder (seq2seq) with Bahdanau attention** (Bahdanau, Cho & Bengio, 2015).

**The plan** 🗺️
1. Load a dataset: real **English → Hinglish** sentences 🇮🇳, or the offline **dates** task (`"friday, 16 october 2026" → "2026-10-16"`).
2. Turn text into token ids (provided).
3. Encoder: a bidirectional GRU (provided).
4. ✍️ **TODO**: `masked_softmax` + `AdditiveAttention.forward` (same names as the `masked_softmax` / `additive_attention` steps on the platform).
5. ✍️ **TODO**: the decoder loop: teacher forcing + gluing the context onto the input.
6. Train with attention 🆚 without attention, compare the loss curves.
7. Score it: exact match (dates) or BLEU-4 (Hinglish), coded from scratch.
8. Look inside the model's head 🧠: attention heatmaps.
9. Translate your own sentence, then 🚀 submit.

**How to use this notebook**
- `Runtime → Change runtime type → T4 GPU`, then fill in the ⚙️ settings cell and `Runtime → Run all`.
- Fill in every cell marked `# TODO` (replace the `...`). The cell right after each one checks your work ✅.
- `QUICK = True` runs a tiny version in ~3 minutes on a CPU (good for debugging; the scores will be low).

In [ ]:
#@title ⚙️ Settings: paste your platform details, pick a dataset
PLATFORM_URL = ""   #@param {type:"string"}
SUBMIT_TOKEN = ""   #@param {type:"string"}
DATASET = "hinglish"  #@param ["hinglish", "dates"]
QUICK = False  #@param {type:"boolean"}

In [ ]:
#@title 🧰 Imports, seed, device
import math, os, random, re, subprocess, sys, textwrap, time
from collections import Counter
from datetime import date, timedelta

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Apple's MPS is skipped on purpose: the step-by-step decoder runs ~10x slower there than on the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  device: {device}   |   dataset: {DATASET}   |   QUICK: {QUICK}")
if device.type != "cuda":
    print("⚠️  No GPU found. Training will be slow. In Colab: Runtime → Change runtime type → T4 GPU "
          "(or set QUICK = True to run a mini version).")

def check(condition, hint):
    """Tiny helper for the ✅ check cells."""
    if not condition:
        raise AssertionError("❌ " + hint)

## 1️⃣ The data

| `DATASET` | input → output | level | notes |
|---|---|---|---|
| `"hinglish"` | `"set an alarm for 7 am"` → `"7 baje ka alarm set karo"` | words | [findnitai/english-to-hinglish](https://huggingface.co/datasets/findnitai/english-to-hinglish) (Apache-2.0), ~189k pairs of chatbot-style commands & questions. We keep short pairs (≤ 12 tokens per side), lowercase, drop duplicate English sentences, and hold out a fixed **2,000-pair test set**. |
| `"dates"` | `"friday, 16 october 2026"` → `"2026-10-16"` | characters | Generated right here, no download. 8 messy formats, including the sneaky month-first `10.16.2026`. Guaranteed to work offline. |

Both datasets end up in the same shape: a list of `(source_text, target_text)` pairs. From there on, **one model and one training loop handle both**.

In [ ]:
#@title 📦 Load the pairs
FORMATS = [
    lambda d: d.strftime("%d %B %Y"),                   # 16 october 2026
    lambda d: d.strftime("%B %d, %Y"),                  # october 16, 2026
    lambda d: d.strftime("%A, %d %B %Y"),               # friday, 16 october 2026
    lambda d: d.strftime("%a %b %d %Y"),                # fri oct 16 2026
    lambda d: f"{d.day} {d.strftime('%b')} {d.year}",   # 16 oct 2026
    lambda d: d.strftime("%d/%m/%Y"),                   # 16/10/2026  (day first)
    lambda d: d.strftime("%m.%d.%Y"),                   # 10.16.2026  (month first!)
    lambda d: f"{d.strftime('%B')} {d.day} {d.year}",   # october 16 2026
]

def make_date_pairs(n, seed):
    rng = random.Random(seed)
    pairs = []
    for _ in range(n):
        d = date(1950, 1, 1) + timedelta(days=rng.randrange(365 * 100))
        pairs.append((rng.choice(FORMATS)(d).lower(), d.isoformat()))
    return pairs

def word_tokenize(text):
    """lowercase, split into words and single punctuation marks: "What's up?" → what ' s up ?"""
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", text.lower())

MAX_WORDS = 12
N_TEST = 2000

if DATASET == "dates":
    train_pairs = make_date_pairs(4000 if QUICK else 20000, seed=0)
    test_pairs = make_date_pairs(500 if QUICK else N_TEST, seed=123)
else:
    try:
        import datasets
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset
    raw = load_dataset("findnitai/english-to-hinglish", split="train")   # one column: "translation" = {"en", "hi_ng", "source"}
    pairs, seen = [], set()
    for row in raw["translation"]:
        en, hi = (row["en"] or "").strip(), (row["hi_ng"] or "").strip()
        en_tok, hi_tok = word_tokenize(en), word_tokenize(hi)
        if not (0 < len(en_tok) <= MAX_WORDS and 0 < len(hi_tok) <= MAX_WORDS):
            continue
        key = " ".join(en_tok)
        if key in seen:              # same English sentence twice → keep one, so the test set never leaks into training
            continue
        seen.add(key)
        pairs.append((" ".join(en_tok), " ".join(hi_tok)))
    random.Random(42).shuffle(pairs)  # fixed shuffle → everyone gets the SAME test set
    test_pairs, train_pairs = pairs[:N_TEST], pairs[N_TEST:]
    if QUICK:
        train_pairs, test_pairs = train_pairs[:15000], test_pairs[:500]
    print(f"kept {len(pairs):,} short pairs out of {len(raw):,}")

print(f"train: {len(train_pairs):,} pairs   test: {len(test_pairs):,} pairs\n")
for src, tgt in random.Random(1).sample(train_pairs, 8):
    print(f"  {src:42s} →  {tgt}")

## 2️⃣ Tokens → ids (provided)

Neural nets eat numbers, not strings. We split each sentence into **tokens** (characters for dates, words for Hinglish) and give every token an id.
Four special tokens:

| token | id | job |
|---|---|---|
| `<pad>` | 0 | filler so sentences in a batch have the same length (the model must **ignore** it) |
| `<sos>` | 1 | "start of sentence": the first input to the decoder |
| `<eos>` | 2 | "end of sentence": the decoder says this when it's done, so outputs can have **any length** |
| `<unk>` | 3 | any word that's not in the vocabulary (we keep the 8,000 most common words per side) |

A batch is `x` (B, T_in) source ids, `lengths` (B,) the real length of each source, and `y` (B, T_out) target ids ending in `<eos>`, padded with `<pad>`.

In [ ]:
#@title 🔤 Vocab + DataLoaders
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]
CHAR_LEVEL = DATASET == "dates"

def tokenize(text):
    return list(text) if CHAR_LEVEL else text.split()   # hinglish text is already word_tokenize'd

def detokenize(tokens):
    return ("" if CHAR_LEVEL else " ").join(tokens)

class Vocab:
    def __init__(self, token_lists, max_size=8000):
        counts = Counter(tok for toks in token_lists for tok in toks)
        self.itos = SPECIALS + [tok for tok, _ in counts.most_common(max_size)]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
    def __len__(self):
        return len(self.itos)
    def encode(self, tokens):
        return [self.stoi.get(tok, UNK) for tok in tokens]
    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS: break
            if i not in (PAD, SOS): out.append(self.itos[i])
        return out

train_tok = [(tokenize(s), tokenize(t)) for s, t in train_pairs]
test_tok = [(tokenize(s), tokenize(t)) for s, t in test_pairs]

def collate(batch):
    """batch: list of (src_tokens, tgt_tokens) → x (B, T_in), lengths (B,), y (B, T_out)"""
    src = [src_vocab.encode(s) for s, _ in batch]
    tgt = [tgt_vocab.encode(t) + [EOS] for _, t in batch]
    x = torch.full((len(batch), max(map(len, src))), PAD, dtype=torch.long)
    y = torch.full((len(batch), max(map(len, tgt))), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(src, tgt)):
        x[i, :len(s)] = torch.tensor(s)
        y[i, :len(t)] = torch.tensor(t)
    return x, torch.tensor([len(s) for s in src]), y

CFG = {   # model size + training budget per dataset
    "dates":    dict(emb=32,  hidden=64,  dropout=0.0, batch=128, lr=2e-3, epochs=8,  base_epochs=4, minutes=4, base_minutes=2, vocab=100),
    "hinglish": dict(emb=128, hidden=256, dropout=0.2, batch=128, lr=1e-3, epochs=10, base_epochs=4, minutes=7, base_minutes=3, vocab=8000),
}[DATASET]
if QUICK:
    CFG.update(epochs=3, base_epochs=2, minutes=1.0, base_minutes=0.4)
    if DATASET == "hinglish":
        CFG.update(emb=64, hidden=96, vocab=3000, minutes=1.3)
    else:
        CFG.update(epochs=6)

src_vocab = Vocab([s for s, _ in train_tok], CFG["vocab"])
tgt_vocab = Vocab([t for _, t in train_tok], CFG["vocab"])
MAX_OUT_LEN = max(len(t) for _, t in train_tok) + 1          # +1 for <eos>; cap for greedy decoding

from torch.utils.data import DataLoader
train_loader = DataLoader(train_tok, batch_size=CFG["batch"], shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_tok, batch_size=256, shuffle=False, collate_fn=collate)

unk_rate = sum(i == UNK for s, _ in test_tok for i in src_vocab.encode(s)) / sum(len(s) for s, _ in test_tok)
print(f"source vocab: {len(src_vocab):,}   target vocab: {len(tgt_vocab):,}   max output length: {MAX_OUT_LEN}")
print(f"test-set source tokens that are <unk>: {unk_rate:.1%}")
x, lengths, y = next(iter(train_loader))
print("batch shapes  x:", tuple(x.shape), " lengths:", tuple(lengths.shape), " y:", tuple(y.shape))
print("\nexample:", train_pairs[0])
print("  x ids:", src_vocab.encode(train_tok[0][0]))
print("  y ids:", tgt_vocab.encode(train_tok[0][1]) + [EOS])

## 3️⃣ The encoder (provided)

A **bidirectional GRU** reads the source left→right *and* right→left, so the vector for each token, `h_j`, knows about the words on both sides. Shape: `(B, T_in, 2H)`.

📦 **Packing.** Sentences in a batch have different lengths, so the short ones are padded with `<pad>`. A plain GRU would happily read those pads, and the backward direction would even *start* on them. `pack_padded_sequence` tells the GRU the true length of every sentence, so it skips the padding completely; `pad_packed_sequence` turns the result back into a normal padded tensor (pad positions are zeros).

The final forward + backward states are squashed through a small `bridge` layer into `h0`, the decoder's starting state: a single **summary vector** of the whole sentence.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb, hidden, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(emb, hidden, batch_first=True, bidirectional=True)
        self.bridge = nn.Linear(2 * hidden, 2 * hidden)   # final fwd+bwd state → decoder h0

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.dropout(self.emb(x)), lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, h = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        h0 = torch.tanh(self.bridge(torch.cat([h[0], h[1]], dim=1)))
        return out, h0   # out: (B, T_in, 2H) one vector per source token;  h0: (B, 2H) summary

_enc = Encoder(len(src_vocab), 8, 16)
_out, _h0 = _enc(x[:4], lengths[:4])
print("encoder out:", tuple(_out.shape), "  h0:", tuple(_h0.shape))

## 4️⃣ ✍️ TODO: Bahdanau (additive) attention

Without attention, the decoder only gets `h0`: the *whole* sentence crushed into one vector (the **bottleneck** 🍾).
With attention, at **every output step** the decoder (state `s`) looks back at **all** encoder vectors `h_1 … h_T`:

$$\text{score}_j = v^\top \tanh(W_{dec}\, s + W_{enc}\, h_j) \qquad \alpha = \text{masked\_softmax}(\text{score}) \qquad \text{context} = \sum_j \alpha_j\, h_j$$

- `masked_softmax`: padding positions must get weight **exactly 0**. Set their score to `-inf` before the softmax (`scores.masked_fill(~mask, float("-inf"))`); `exp(-inf) = 0` 🎯. (PyTorch's `softmax` already subtracts the max for numerical stability, so you don't need to.)
- Batched shapes: `W_enc(enc_out)` is `(B, T, A)`, `W_dec(s)` is `(B, A)`. Add a middle dimension with `.unsqueeze(1)` → `(B, 1, A)` so it **broadcasts** over all `T` positions. After `self.v(...)` you have `(B, T, 1)`: `.squeeze(-1)` it.
- The context is a weighted average of the encoder vectors: `torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)` or `(weights.unsqueeze(-1) * enc_out).sum(dim=1)`.

In [ ]:
def masked_softmax(scores, mask):
    """scores: (B, T) floats;  mask: (B, T) bool, True = real token.
    Returns weights (B, T): every row sums to 1 and padding gets exactly 0."""
    # TODO 1: masked-out positions get -inf, then softmax over the last dimension
    return ...


class AdditiveAttention(nn.Module):
    """score(s, h_j) = vᵀ tanh(W_dec s + W_enc h_j)   (Bahdanau et al., 2015)"""

    def __init__(self, enc_dim, dec_dim, attn_dim=64):
        super().__init__()
        self.W_enc = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_dec = nn.Linear(dec_dim, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, dec_h, enc_out, mask):
        """dec_h: (B, D) decoder state s;  enc_out: (B, T, E) encoder vectors h_j;  mask: (B, T) bool.
        Returns context (B, E) and weights (B, T)."""
        # TODO 2: one score per source position, shape (B, T)
        scores = ...
        weights = masked_softmax(scores, mask)
        # TODO 3: the context vector = weighted average of the encoder vectors, shape (B, E)
        context = ...
        return context, weights

In [ ]:
#@title ✅ Check: attention
torch.manual_seed(0)
_mask = torch.tensor([[True, True, True, False, False], [True, True, True, True, True]])
_w = masked_softmax(torch.randn(2, 5), _mask)
check(isinstance(_w, torch.Tensor), "masked_softmax should return a tensor. Did you replace the `...` (TODO 1)?")
check(_w.shape == (2, 5), f"masked_softmax: expected shape (2, 5), got {tuple(_w.shape)}. Softmax over dim=-1.")
check(torch.allclose(_w.sum(-1), torch.ones(2)), "masked_softmax: each row of weights must sum to 1 (softmax over dim=-1).")
check((_w[0, 3:] == 0).all(), "masked_softmax: padding must get weight exactly 0. Fill masked scores with float('-inf') BEFORE the softmax.")
check(not torch.isnan(masked_softmax(torch.tensor([[1000.0, 1001.0]]), torch.tensor([[True, True]]))).any(),
      "masked_softmax returned nan for big scores.")

_att = AdditiveAttention(enc_dim=6, dec_dim=4, attn_dim=8)
_s, _H = torch.randn(2, 4), torch.randn(2, 5, 6)
try:
    _ctx, _w = _att(_s, _H, _mask)
except (TypeError, AttributeError, RuntimeError) as e:
    raise AssertionError(f"❌ AdditiveAttention.forward crashed ({e}). Fill in `scores` (TODO 2) and `context` (TODO 3); check the shapes in the hints above.")
check(isinstance(_ctx, torch.Tensor) and isinstance(_w, torch.Tensor), "AdditiveAttention: fill in `scores` (TODO 2) and `context` (TODO 3).")
check(_w.shape == (2, 5), f"weights should be (B, T) = (2, 5), got {tuple(_w.shape)}. Did you .unsqueeze(1) W_dec(s) and .squeeze(-1) after v?")
check(_ctx.shape == (2, 6), f"context should be (B, E) = (2, 6), got {tuple(_ctx.shape)}.")
check(torch.allclose(_w.sum(-1), torch.ones(2), atol=1e-5) and (_w[0, 3:] == 0).all(), "weights must sum to 1 and be 0 on padding.")
with torch.no_grad():   # slow, obvious reference: loop over every batch item and position
    _ref = torch.tensor([[_att.v(torch.tanh(_att.W_dec(_s[b]) + _att.W_enc(_H[b, j]))).item() for j in range(5)] for b in range(2)])
    _ref_w = _ref.masked_fill(~_mask, float("-inf")).softmax(-1)
    _ref_ctx = (_ref_w.unsqueeze(-1) * _H).sum(1)
check(torch.allclose(_w, _ref_w, atol=1e-5), "weights don't match vᵀ tanh(W_dec s + W_enc h_j). Check the formula in TODO 2.")
check(torch.allclose(_ctx, _ref_ctx, atol=1e-5), "context should be Σ_j weights_j · h_j (TODO 3).")
print("✅ nice! Your attention computes the right weights, ignores padding, and builds the context vector.")

## 5️⃣ ✍️ TODO: the decoder loop

The decoder writes the output **one token at a time** with a `GRUCell`. At each step `t`:
1. Embed the previous token (`<sos>` at the start).
2. 🔍 Attention: compare the current state `h` with every encoder vector → `context`.
3. **TODO 4**: the GRU cell's input is the token embedding **and** the context, glued side by side: `torch.cat([emb, context], dim=1)` → `(B, emb + 2H)`.
4. Update the state, predict a distribution over the target vocab (`step_logits`).
5. **TODO 5**: choose the next input token:
   - 👩‍🏫 **training** (`target` given) → **teacher forcing**: feed the TRUE previous token `target[:, t]`, even if the model guessed wrong. Training is faster and stabler because one early mistake can't derail the rest of the sentence.
   - 🤖 **inference** (`target is None`) → feed the model's own best guess: `step_logits.argmax(dim=1)`.

With `use_attention=False` the same class becomes the **no-attention baseline**: the decoder only ever sees `h0`.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, src_size, tgt_size, emb=32, hidden=64, use_attention=True, dropout=0.0):
        super().__init__()
        self.use_attention = use_attention
        enc_dim = dec_dim = 2 * hidden
        ctx_dim = enc_dim if use_attention else 0
        self.encoder = Encoder(src_size, emb, hidden, dropout)
        self.out_emb = nn.Embedding(tgt_size, emb, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.cell = nn.GRUCell(emb + ctx_dim, dec_dim)
        self.attention = AdditiveAttention(enc_dim, dec_dim, attn_dim=hidden) if use_attention else None
        self.out = nn.Linear(dec_dim + ctx_dim, tgt_size)

    def forward(self, x, lengths, target=None, max_len=None):
        """target given → teacher forcing, returns logits (B, T_out, V).
        target None → greedy decoding for up to max_len steps (stops early once every row said <eos>).
        Also returns the attention weights (B, steps, T_in) (None for the baseline)."""
        enc_out, h = self.encoder(x, lengths)
        mask = x != PAD
        steps = target.size(1) if target is not None else (max_len or MAX_OUT_LEN)
        token = torch.full((x.size(0),), SOS, dtype=torch.long, device=x.device)
        finished = torch.zeros(x.size(0), dtype=torch.bool, device=x.device)
        logits, attn = [], []
        for t in range(steps):
            emb = self.dropout(self.out_emb(token))                  # (B, emb)
            if self.use_attention:
                context, w = self.attention(h, enc_out, mask)       # (B, 2H), (B, T_in)
                # TODO 4: glue the embedding and the context together → (B, emb + 2H)
                inp = ...
                h = self.cell(inp, h)
                step_logits = self.out(torch.cat([h, context], dim=1))
                attn.append(w)
            else:
                h = self.cell(emb, h)
                step_logits = self.out(h)
            logits.append(step_logits)
            # TODO 5: next input token: the TRUE token target[:, t] when training, else the model's argmax guess
            token = ...
            if target is None:
                finished |= token == EOS
                if finished.all():
                    break
        attn = torch.stack(attn, dim=1) if attn else None
        return torch.stack(logits, dim=1), attn

In [ ]:
#@title ✅ Check: decoder
torch.manual_seed(0)
_m = Seq2Seq(src_size=11, tgt_size=7, emb=8, hidden=16, use_attention=True).eval()
_x = torch.randint(4, 11, (3, 6)); _x[1, 4:] = PAD; _x[2, 2:] = PAD
_len = torch.tensor([6, 4, 2])
_y = torch.randint(4, 7, (3, 5))
try:
    _logits, _attn = _m(_x, _len, target=_y)
except (TypeError, AttributeError) as e:
    raise AssertionError(f"❌ The decoder crashed ({e}). Did you fill in TODO 4 (`inp`) and TODO 5 (`token`)? (The attention check above must pass first.)")
check(_logits.shape == (3, 5, 7), f"logits should be (B, T_out, V) = (3, 5, 7), got {tuple(_logits.shape)}")
check(_attn.shape == (3, 5, 6), f"attention should be (B, T_out, T_in) = (3, 5, 6), got {tuple(_attn.shape)}")
check((_attn[1, :, 4:] == 0).all() and (_attn[2, :, 2:] == 0).all(), "attention leaks onto padding.")

_y2 = _y.clone(); _y2[:, 2] = 4 + (_y[:, 2] - 4 + 1) % 3     # change ONLY the true token at position 2
with torch.no_grad():
    _logits2, _ = _m(_x, _len, target=_y2)
check(torch.allclose(_logits[:, :3], _logits2[:, :3]), "steps 0-2 changed when only target[:, 2] changed. Something is peeking ahead.")
check(not torch.allclose(_logits[:, 3], _logits2[:, 3]),
      "teacher forcing: with `target` given, the input at step t+1 must be target[:, t] (TODO 5).")

with torch.no_grad():
    _g, _ = _m(_x, _len, max_len=5)
    _tf, _ = _m(_x, _len, target=_g.argmax(-1))
check(_g.shape[1] <= 5, "greedy decoding should run at most max_len steps.")
check(torch.allclose(_g, _tf, atol=1e-5),
      "greedy decoding: with target=None the next input must be the model's own guess step_logits.argmax(dim=1) (TODO 5).")
print("✅ nice! Teacher forcing while training, own guesses at test time, and the context is in the loop.")

## 6️⃣ Train: attention 🆚 no attention

Same data, same encoder, same decoder. The **only** difference is whether the decoder can look back at the source.
The loss is cross-entropy over the target tokens (`ignore_index=PAD`, so padding doesn't count).
We give the baseline a smaller budget; it's here to show the gap, not to win.

⏱️ Each model trains for its epochs **or** its time budget, whichever runs out first (roughly 10 minutes total on a T4 with `QUICK = False`).

In [ ]:
def run_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x, lengths, target=y)
            total += F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=PAD, reduction="sum").item()
            n += (y != PAD).sum().item()
    model.train()
    return total / n

def train_model(use_attention, epochs, minutes):
    name = "attention" if use_attention else "no attention"
    torch.manual_seed(SEED)
    model = Seq2Seq(len(src_vocab), len(tgt_vocab), CFG["emb"], CFG["hidden"], use_attention, CFG["dropout"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    hist = {"train": [], "test": []}
    start = time.time()
    for epoch in range(1, epochs + 1):
        total, n = 0.0, 0
        for x, lengths, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x, lengths, target=y)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=PAD)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item(); n += 1
            if time.time() - start > minutes * 60:     # budget is checked every batch
                break
        hist["train"].append(total / n)
        hist["test"].append(run_loss(model, test_loader))
        elapsed = time.time() - start
        print(f"  {name:12s} epoch {epoch:2d}  train loss {hist['train'][-1]:.3f}  test loss {hist['test'][-1]:.3f}  ({elapsed:.0f}s)")
        if elapsed > minutes * 60 and epoch < epochs:
            print(f"  ⏱️ time budget of {minutes} min used up, stopping early")
            break
    return model, hist

print(f"🏋️ training WITH attention ({sum(p.numel() for p in Seq2Seq(len(src_vocab), len(tgt_vocab), CFG['emb'], CFG['hidden']).parameters()):,} params)")
model, hist_attn = train_model(True, CFG["epochs"], CFG["minutes"])
print("\n🏋️ training the no-attention baseline")
baseline, hist_base = train_model(False, CFG["base_epochs"], CFG["base_minutes"])

In [ ]:
#@title 📉 Loss curves
fig, ax = plt.subplots(figsize=(7, 4))
for hist, label, color in [(hist_attn, "with attention", "tab:blue"), (hist_base, "no attention (bottleneck)", "tab:orange")]:
    ep = range(1, len(hist["train"]) + 1)
    ax.plot(ep, hist["train"], "--", color=color, alpha=0.6, label=f"{label}: train")
    ax.plot(ep, hist["test"], "-o", color=color, label=f"{label}: test")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy per token"); ax.set_title(f"{DATASET}: attention vs bottleneck")
ax.legend(); plt.tight_layout(); plt.show()

## 7️⃣ How good is it? 📏

We translate the test set with **greedy decoding** (always take the most likely next token) and score it:

- **dates** → **exact match**: % of test dates where the whole output string is exactly right. One wrong digit = wrong.
- **hinglish** → **BLEU-4**, the classic machine-translation score (0–100). For every n = 1…4 it counts how many of the model's n-grams also appear in the reference (clipped, so repeating "hai hai hai" doesn't farm points), takes the geometric mean of those 4 precisions, and multiplies by a **brevity penalty** `exp(1 − ref_len / hyp_len)` when the output is too short (otherwise a 1-word answer would score great precision). Computed over the whole corpus at once, in a few lines of plain Python below 👇

🤔 **What to expect for Hinglish.** It's a small model trained for a few minutes on ~120k sentences, with only **one** reference translation per sentence (and Hinglish spelling is wild: *nahi / nahin / nhi*). In our test run the attention model reached **BLEU ≈ 35** after less than one epoch (the no-attention baseline ≈ 20), so on a T4 expect somewhere in the **30s–40s**. It gets the gist and the Hindi word order of short commands right (*"cancel 1 hour alarm" → "1 hour alarm ko cancel kare"*) but fumbles rare words and names (they become `<unk>` or get swapped for a common word). As a sanity check we also score a **trivial baseline**: just copy the English input. Hinglish borrows lots of English words (alarm, traffic, play…) so copying scores ≈ 8, not 0; the model should beat it by a mile. With `QUICK = True` (tiny model, ~1 minute of training) expect single digits: it only learns the most common patterns, so it may even lose to the copy baseline.

In [ ]:
@torch.no_grad()
def translate_pairs(model, tok_pairs, batch_size=256):
    """greedy-decode every source → list of output token lists (+ attention, one (steps, T_in) array per sentence)"""
    model.eval()
    outs, attns = [], []
    for i in range(0, len(tok_pairs), batch_size):
        x, lengths, _ = collate(tok_pairs[i:i + batch_size])
        logits, attn = model(x.to(device), lengths, max_len=MAX_OUT_LEN)
        ids = logits.argmax(-1).cpu().tolist()
        for b, row in enumerate(ids):
            outs.append(tgt_vocab.decode(row))
            if attn is not None:
                attns.append(attn[b, :, :lengths[b]].cpu().numpy())
    model.train()
    return outs, attns

def corpus_bleu(hyps, refs, max_n=4):
    """hyps, refs: lists of token lists. Returns BLEU-4 in [0, 100]."""
    matches, totals = [0] * max_n, [0] * max_n
    hyp_len = ref_len = 0
    for h, r in zip(hyps, refs):
        hyp_len += len(h); ref_len += len(r)
        for n in range(1, max_n + 1):
            h_ngrams = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1))
            r_ngrams = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
            matches[n - 1] += sum(min(c, r_ngrams[g]) for g, c in h_ngrams.items())   # clipped counts
            totals[n - 1] += max(len(h) - n + 1, 0)
    if min(matches) == 0:
        return 0.0
    log_precision = sum(math.log(m / t) for m, t in zip(matches, totals)) / max_n
    brevity_penalty = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    return 100 * brevity_penalty * math.exp(log_precision)

assert abs(corpus_bleu([["a", "b", "c", "d"]], [["a", "b", "c", "d"]]) - 100) < 1e-9   # perfect = 100

refs = [t for _, t in test_tok]
preds, attns = translate_pairs(model, test_tok)
base_preds, _ = translate_pairs(baseline, test_tok)

if DATASET == "dates":
    METRIC_NAME = "exact_match"
    score = lambda hyps: 100 * sum(h == r for h, r in zip(hyps, refs)) / len(refs)
    METRIC, base_score = score(preds), score(base_preds)
    print(f"🎯 exact match   with attention: {METRIC:.1f}%    no attention: {base_score:.1f}%")
else:
    METRIC_NAME = "bleu"
    METRIC, base_score = corpus_bleu(preds, refs), corpus_bleu(base_preds, refs)
    copy_score = corpus_bleu([s for s, _ in test_tok], refs)
    print(f"🎯 BLEU-4   with attention: {METRIC:.1f}    no attention: {base_score:.1f}    copy-the-English baseline: {copy_score:.1f}")

print("\n10 test translations:")
for i in range(10):
    src, ref = test_pairs[i]
    mark = "✅" if preds[i] == refs[i] else "  "
    print(f"{mark} {src}\n     model: {detokenize(preds[i])}\n     ref:   {ref}\n     base:  {detokenize(base_preds[i])}")

## 8️⃣ Peek inside: attention heatmaps 🔥

Each heatmap is one translation. **Rows** = output tokens the model wrote (top to bottom), **columns** = source tokens. Row `t` is the weight vector `α` from step `t`: bright = "I was looking here when I wrote this token". Every row sums to 1.

How to read them:
- **dates**: the year digits light up on the year in the input, `-10-` on "october" or on the `10` in `10.16.2026`. For `16/10/2026` vs `10.16.2026`, the model has learned to look at *different* positions for the month, depending on the separator. 🤯
- **hinglish**: English is subject–verb–object, Hindi is subject–object–**verb**. So you'll often see the diagonal **bend**: "set an alarm for 7 am" → "7 baje ka alarm **set karo**", the verb moves to the end and the attention follows it. Borrowed English words (alarm, traffic) usually get a sharp bright square.
- A blurry row = the model wasn't sure where to look (common for function words like *ka / ki / hai*, which have no single English match).

In [ ]:
#@title 🔥 Heatmaps → result.png
def plot_attention(ax, src_tokens, out_tokens, weights):
    rows = min(len(out_tokens) + 1, len(weights))          # output tokens + the final <eos> step
    ax.imshow(weights[:rows], cmap="viridis", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens, rotation=60 if not CHAR_LEVEL else 0, fontsize=9)
    ax.set_yticks(range(rows)); ax.set_yticklabels((out_tokens + ["<eos>"])[:rows], fontsize=9)
    ax.set_xlabel("source"); ax.set_ylabel("output")
    ax.set_title(textwrap.fill(detokenize(src_tokens), 50) + "\n→ " + textwrap.fill(detokenize(out_tokens), 50), fontsize=10)

# 3 test examples of medium length (long enough to be interesting, short enough to read)
cand = [i for i, (s, t) in enumerate(test_tok) if 5 <= len(s) <= 9] if not CHAR_LEVEL else list(range(len(test_tok)))
show = cand[:3]
fig, axes = plt.subplots(1, 3, figsize=(17, 5.8))
for ax, i in zip(axes, show):
    plot_attention(ax, test_tok[i][0], preds[i], attns[i])
fig.suptitle(f"Bahdanau attention on {DATASET} · {METRIC_NAME} = {METRIC:.1f}", fontsize=12)
plt.tight_layout()
plt.savefig("result.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"saved result.png ({os.path.getsize('result.png') / 1e6:.2f} MB)")

## 9️⃣ Translate your own sentence 🎤

Type anything in `MY_SENTENCE` (leave it empty for a default). Tips: keep it short, chatbot-ish (alarms, weather, music, traffic, reminders…) because that's what the data is about. Words the model never saw become `<unk>`.
For `dates`, try any of the formats, e.g. `"25 dec 1999"` or `"07.04.1976"`.

In [ ]:
MY_SENTENCE = ""  #@param {type:"string"}

def translate(text, show=True):
    src = list(text.lower()) if CHAR_LEVEL else word_tokenize(text)
    src = [tok for tok in src if tok in src_vocab.stoi or not CHAR_LEVEL]   # drop unknown characters for dates
    (out,), (weights,) = translate_pairs(model, [(src, [])])
    print(f"{text}  →  {detokenize(out)}")
    if show:
        fig, ax = plt.subplots(figsize=(max(5, len(src) * (0.35 if CHAR_LEVEL else 0.7)), 0.45 * len(out) + 2))
        plot_attention(ax, [tok if tok in src_vocab.stoi else "<unk>" for tok in src], out, weights)
        plt.tight_layout(); plt.show()
    return detokenize(out)

if DATASET == "dates":
    examples = ["saturday, 1 january 2000", "25 dec 1999", "07.04.1976"]
else:
    examples = ["set an alarm for 7 am tomorrow", "what is the weather like today", "play some music by arijit singh"]
for s in examples[1:]:
    translate(s, show=False)
translate(MY_SENTENCE or examples[0]);

## 🚀 Submit

Run the cell below. It sends your score (`exact_match` % for dates, `bleu` for Hinglish) and your `result.png` heatmaps to DL Lab. Make sure `PLATFORM_URL` and `SUBMIT_TOKEN` are filled in at the top.

🧪 **Want a better score?** Try more epochs, a bigger `hidden`, a larger vocab, or beam search instead of greedy decoding. Write what you tried in `notes`!

In [ ]:
#@title 🚀 Submit your result to DL Lab
import base64, json, urllib.request, urllib.error
CHALLENGE_ID = "attention"   # "sequences" | "attention" | "gans"

def submit(metric_name=None, metric=None, image_path=None, notes=""):
    if not PLATFORM_URL or not SUBMIT_TOKEN:
        print("⚠️ Paste PLATFORM_URL and SUBMIT_TOKEN into the settings cell at the top (copy them from the platform's challenge page).")
        return
    body = {"dataset": DATASET, "metric_name": metric_name, "metric": None if metric is None else float(metric), "notes": notes}
    if image_path:
        with open(image_path, "rb") as f:
            body["image_b64"] = base64.b64encode(f.read()).decode()
    req = urllib.request.Request(PLATFORM_URL.rstrip("/") + f"/api/challenges/{CHALLENGE_ID}/submit",
                                 data=json.dumps(body).encode(), method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer " + SUBMIT_TOKEN.strip()})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            print("🎉", json.loads(r.read())["message"])
    except urllib.error.HTTPError as e:
        print("❌ Submit failed:", e.code, e.read().decode()[:300])
    except Exception as e:
        print("❌ Could not reach the platform:", e)

submit(metric_name="exact_match" if DATASET == "dates" else "bleu", metric=METRIC, image_path="result.png",
       notes=f"seq2seq + Bahdanau attention, hidden={CFG['hidden']}, {len(hist_attn['train'])} epochs, QUICK={QUICK}; "
             f"no-attention baseline scored {base_score:.1f}")